In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

In [ ]:
!pip show cellpose

In [ ]:
import numpy as np
np.__version__

### Install Cellpose-SAM

https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb

In [ ]:
import numpy as np
import os, sys
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
from natsort import natsorted

### Faltou:

sudo apt install nvidia-cuda-toolkit


In [ ]:
!lspci | grep -i nvidia

In [ ]:
!nvcc --version

In [ ]:
!nvidia-smi

In [ ]:
# reinstall cellpose to see currect nvidia version - is necessary?
# !pip3 install cellpose --force
# Successfully uninstalled cellpose-4.0.7

In [ ]:
import torch
torch.__version__, np.__version__

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.device_count()

In [ ]:
torch.cuda.get_device_name(0), torch.version.cuda

In [ ]:
# !pip3 install cellpose --force

In [ ]:
from cellpose import models, core, io, plot

In [ ]:
# !pip install "pillow<10"
# !pip install pyqtgraph==0.11.1
# !pip install pyqt5==5.15.4
# !pip3 install numpy==1.23.5

In [ ]:
import PIL
PIL.__version__

In [ ]:
# !pip3 install PyQt5

In [ ]:
import PyQt5
PyQt5

In [ ]:
# !cellpose --verbose

#### run this to get printing of progress

In [ ]:
io.logger_setup()

#### Check if colab notebook instance has GPU access

In [ ]:
if core.use_gpu()==False:
    print(ImportError("No GPU access, change your runtime"))
else:
    print("GPU Ok")

### Testing Matrix multiplication

In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

B, H, N, D = 2, 4, 128, 64
q = torch.randn(B, H, N, D, device='cuda', dtype=torch.float32)
k = torch.randn(B, H, N, D, device='cuda', dtype=torch.float32)
scale = (D ** -0.5)

try:
    out = (q * scale) @ k.transpose(-2, -1)
    print("FP32 batched matmul OK:", out.shape)
except Exception as e:
    print("FP32 batched matmul FAILED:", repr(e))

### CellposeModel

  - folder: /home/flalix/.cellpose/models/

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model="cyto2")

In [ ]:
is_laptop = True

if is_laptop:
    root_image = "../../colaboracoes/carlos_deOcesano"
else:
    root_image = "/media/flalix/d2f268d1-512d-499f-b3b3-6dad7d3fdd25/colaboracoes/deOcesano"

os.listdir(root_image)

In [ ]:
root_hcs = os.path.join(root_image, 'Plate1848')
hcs_folders = os.listdir(root_hcs)
print(root_hcs)
hcs_folders

In [ ]:
root_1perc = os.path.join(root_hcs, '1% SFB')
os.path.exists(root_1perc), root_1perc

In [ ]:
files = os.listdir(root_1perc)
for fname in files[:5]:
    print(fname)

In [ ]:
i=0
filefig = os.path.join(root_1perc, files[i])
img = io.imread(filefig)

fig = plt.figure(figsize=(8, 8))
plt.imshow(img)

In [ ]:
type(img)

In [ ]:
img.shape

In [ ]:
type(img[:,0,0])

In [ ]:
[type(x) for x in img[:5,0,0]]

In [ ]:
[type(x) for x in img[0,:5,0]]

In [ ]:
first_channel = '0' # @param ['None', 0, 1, 2, 3, 4, 5]
second_channel = '1' # @param ['None', 0, 1, 2, 3, 4, 5]
third_channel = '2' # @param ['None', 0, 1, 2, 3, 4, 5]

In [ ]:
selected_channels = []
for i, c in enumerate([first_channel, second_channel, third_channel]):
  if c == 'None':
    continue
  if int(c) > img.shape[-1]:
    assert False, 'invalid channel index, must have index greater or equal to the number of channels'
  if c != 'None':
    selected_channels.append(int(c))

selected_channels

In [ ]:
%%time

img_selected_channels = np.zeros_like(img)
img_selected_channels[:, :, :len(selected_channels)] = img[:, :, selected_channels]

flow_threshold = 0.4
cellprob_threshold = 0.0
tile_norm_blocksize = 0

masks, flows, styles = model.eval(
    img_selected_channels,
    batch_size=32,
    flow_threshold=flow_threshold,
    cellprob_threshold=cellprob_threshold,
    normalize={"tile_norm_blocksize": tile_norm_blocksize}
)

In [ ]:
print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))

In [ ]:
fig = plt.figure(figsize=(12,12))
plot.show_segmentation(fig, img_selected_channels, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
del(model)

In [ ]:
del(models)

In [ ]:
del(core)

In [ ]:
del(io)

In [ ]:
del(plot)